In [1]:
import os
import glob
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.sparse as sp
from scipy.stats import pearsonr
from tqdm import tqdm
import time
from collections import Counter
from matplotlib.patches import Patch
from scipy.spatial.distance import cdist
import re
import warnings

from scipy.stats import zscore
from matplotlib.colors import LinearSegmentedColormap
from scipy.cluster.hierarchy import linkage, fcluster
from collections import defaultdict

import rmm
import cupy
import cudf
import cupy as cp
import cvxpy as cpv
from rmm.allocators.cupy import rmm_cupy_allocator
from sklearn.feature_extraction.text import CountVectorizer
from cuml.linear_model import ElasticNet
import anndata as an
import scanpy as sc
import rapids_singlecell as rsc

import torch
from torchnmf.nmf import NMF

# Enable `managed_memory`
rmm.reinitialize(
    managed_memory=True,
    pool_allocator=False,
)
cp.cuda.set_allocator(rmm_cupy_allocator)

CUDARuntimeError: cudaErrorDevicesUnavailable: CUDA-capable device(s) is/are busy or unavailable

In [ ]:
%%time
def fit_elasticnet_deconvolution(
    rdata, sdata,
    alpha=0.1,
    l1_ratio=1.0,
    fit_intercept=False,
    norm_weights=True,
    max_iter=1000
):
    """
    Fit ElasticNet models to deconvolve each sample in sdata using rdata as reference.

    Returns:
        weights_df: DataFrame [cells x ref_cell_types] of weights
        errors: list of MSEs for each cell
    """
    # Reference matrix and metadata
    R = rdata.X.toarray() if hasattr(rdata.X, "toarray") else np.array(rdata.X)
    ref_cell_types = rdata.obs['cell_type'].tolist()
    # Target matrix
    A = sdata.X.toarray() if hasattr(sdata.X, "toarray") else np.array(sdata.X)
    target_names = sdata.obs_names.tolist()

    weights = []
    errors = []

    for i in range(A.shape[0]):
        x = A[i]
        model = ElasticNet(
            alpha=alpha,
            l1_ratio=l1_ratio,
            fit_intercept=fit_intercept,
            max_iter=max_iter
        )
        model.fit(R.T, x)
        w = model.coef_
        if norm_weights:
            w = w / np.sum(np.abs(w))
        weights.append(w)
        x_pred = np.dot(R.T, w)
        mse = np.mean((x - x_pred) ** 2)
        errors.append(mse)

    weights_df = pd.DataFrame(weights, columns=ref_cell_types, index=target_names)
    return weights_df, errors